# Agentic RAG — Retrieval as a Tool in a ReAct Loop
## Retrieval as a Tool — The ReAct + RAG Fusion
⏱ ~12 min

**Agentic RAG** is the evolution of pipeline RAG: instead of always retrieving on every query,
an LLM agent decides *whether* to retrieve, *what* to retrieve, and *which source* to hit —
all at runtime. The agent may retrieve once, twice, from multiple sources, or not at all.

By the end of this session you will understand *why* agentic RAG outperforms pipeline RAG
for complex questions, *how* the ReAct loop works, and *how* to build a multi-tool
retrieval agent with LangGraph in under 100 lines of code.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Pipeline RAG vs Agentic RAG; the ReAct loop |
| 2 | **The Knowledge Base** — Building and querying a Chroma vector store |
| 3 | **Tools** — `@tool` decorator; three tools: vectorstore, web, calculator |
| 4 | **The Agent** — `create_react_agent`; system prompt design |
| 5 | **Running & Tracing** — Mixed questions; inspecting the full ReAct trace |
| 6 | **Multi-Hop Reasoning** — Questions that require two tool calls in sequence |
| 7 | **Debugging & Safety** — Prompt ablations; `eval()` sandbox; failure modes |
| ★ | **Extensions** (bonus) — Adding tools; streaming; memory |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Companion Examples
> **4-basic-react-agent** — ReAct basics with a simpler tool set  
> **25-adaptive-rag** — rule-based routing vs agent-driven tool choice  
> **12-basic-rag-notebook** — pipeline RAG foundation (read this first if RAG is new to you)  
> **11-hitl-approval** — pause the agent mid-run for human review

### Key References
> Yao, S., Zhao, J., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629  
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401  
> Trivedi, H., Balasubramanian, N., et al. (2022). *Interleaving Retrieval with Chain-of-Thought Reasoning for Knowledge-Intensive Multi-Step Questions (IRCoT).* https://arxiv.org/abs/2212.10509  
> LangGraph `create_react_agent` docs: https://langchain-ai.github.io/langgraph/reference/prebuilt/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "langgraph",
            "chromadb",
            "duckduckgo-search",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab:  set in Secrets panel (left sidebar key icon) as  OPENAI_API_KEY
# Local:  create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or looks wrong. "
    "Local: add it to .env. Colab: add it in the Secrets panel."
)
print(f"OPENAI_API_KEY loaded: {key[:8]}...{key[-4:]}")

---

## Part 1 — Pipeline RAG vs Agentic RAG

---

### The Limitation of Pipeline RAG

In pipeline RAG (example 12) every query goes through a fixed sequence:

```
Query → Embed → Retrieve top-k → Stuff into prompt → LLM → Answer
```

This works well for simple, single-hop lookups. It breaks down when:
- The question needs **multiple retrieval steps** (multi-hop reasoning)
- The answer requires **different sources** (internal KB + live web)
- Retrieval is **unnecessary** (math, greetings) but still fires, wasting tokens
- The query is **ambiguous** and needs clarification before retrieval

---

### The Agentic RAG Solution

Agentic RAG gives the LLM retrieval as a *tool* rather than a *pipeline stage*.
The agent reasons about each question and decides what to do next.

```
╔══════════════════════════════════════════════════════════════════╗
║                    AGENTIC RAG — ReAct Loop                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║   User Question                                                  ║
║        │                                                         ║
║        ▼                                                         ║
║   ┌─────────┐   THINK: What do I need to answer this?           ║
║   │  Agent  │──────────────────────────────────┐                ║
║   │  (LLM)  │                                  │                ║
║   └─────────┘                                  ▼                ║
║        ▲               ┌──────────────────────────────────┐     ║
║        │               │          Tool Selection           │     ║
║        │               ├──────────────────────────────────┤     ║
║        │               │  vectorstore_search               │     ║
║   OBSERVE              │   └─ internal Chroma KB           │     ║
║   (tool result)        │                                   │     ║
║        │               │  web_search                       │     ║
║        │               │   └─ DuckDuckGo live results      │     ║
║        │               │                                   │     ║
║        │               │  calculator                       │     ║
║        │               │   └─ Python eval (sandboxed)      │     ║
║        │               │                                   │     ║
║        │               │  [direct answer]                  │     ║
║        │               │   └─ no tool needed               │     ║
║        └───────────────┴──────────────────────────────────┘     ║
║                                                                  ║
║        ▼  (loop until agent decides it has enough context)       ║
║   Final Answer                                                   ║
╚══════════════════════════════════════════════════════════════════╝
```

This is the **ReAct pattern** (Yao et al. 2023): **Re**asoning + **Act**ing interleaved.
The agent alternates between thinking about what to do and taking actions (tool calls).

---

### Pipeline RAG vs Agentic RAG — At a Glance

| Dimension | Pipeline RAG | Agentic RAG |
|-----------|-------------|-------------|
| **Retrieval trigger** | Always, every query | Only when the agent decides it's needed |
| **Number of retrieval steps** | Fixed (1) | Dynamic (0, 1, 2, …) |
| **Sources** | One vector store | Any tool (KB, web, DB, API, …) |
| **Multi-hop reasoning** | Not supported | Native — agent loops until satisfied |
| **Complexity** | Low — just a chain | Higher — needs a graph runtime |
| **Latency** | Predictable | Variable (more tool calls = more latency) |
| **Cost** | Fixed per query | Scales with reasoning steps |
| **Best for** | Simple Q&A over one corpus | Complex, heterogeneous, multi-step queries |

---

### Tool Selection Strategies

| Strategy | How the agent picks a tool | Example |
|----------|---------------------------|--------|
| **LLM-driven** (this notebook) | Model reads tool docstrings and decides | `vectorstore_search` for internal docs |
| **Rule-based routing** | Classifier pre-routes query by category | If query contains "current" → web |
| **Confidence threshold** | Retrieve only if retrieval score < threshold | See example 25-adaptive-rag |
| **Self-RAG tokens** | Model generates reflection tokens (retrieve/no-retrieve) | See Self-RAG paper (Asai et al. 2023) |

---

## Part 2 — Building the Knowledge Base

---

The internal knowledge base is a Chroma vector store seeded with facts about LangGraph and RAG patterns.
When the agent calls `vectorstore_search`, this is what it searches.

In a production system this KB would be loaded from real documents (PDFs, wikis, databases).
Here we seed it inline so the notebook is self-contained.

### What Makes a Good KB for Agentic RAG?

| Concern | Pipeline RAG | Agentic RAG |
|---------|-------------|-------------|
| **Chunk size** | 300–600 chars (tuned for k=3–5) | Shorter (150–300) — agent may call multiple times |
| **Metadata** | Optional | Critical — agent can filter by source/topic |
| **Coverage gaps** | Answered by the LLM's training | Agent falls through to web search |
| **Freshness** | Rebuilt on doc update | Agent can always supplement with web_search |

In [ ]:
# ─── 2-A: Seed the knowledge base ─────────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

KB_DOCS = [
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs.",
        metadata={"topic": "langgraph", "source": "internal-kb"},
    ),
    Document(
        page_content="RAG combines a retriever with a generative model to ground answers in source documents.",
        metadata={"topic": "rag", "source": "internal-kb"},
    ),
    Document(
        page_content="Self-RAG uses reflection tokens to decide whether retrieval is needed per query. "
                     "Reflection tokens include Retrieve, ISREL, ISSUP, and ISUSE.",
        metadata={"topic": "self-rag", "source": "internal-kb"},
    ),
    Document(
        page_content="Corrective RAG (CRAG) adds a retrieval evaluator: if retrieved docs are irrelevant, "
                     "it rewrites the query and searches the web before generating.",
        metadata={"topic": "corrective-rag", "source": "internal-kb"},
    ),
    Document(
        page_content="Human-in-the-loop checkpointing pauses graph execution at a defined node "
                     "and waits for human approval before proceeding.",
        metadata={"topic": "hitl", "source": "internal-kb"},
    ),
    Document(
        page_content="Adaptive RAG classifies each incoming query and routes it to the cheapest "
                     "correct retrieval path: no-retrieval, single-step RAG, or multi-step RAG.",
        metadata={"topic": "adaptive-rag", "source": "internal-kb"},
    ),
    Document(
        page_content="IRCoT (Interleaving Retrieval with Chain-of-Thought) interleaves retrieval "
                     "steps with chain-of-thought reasoning to answer multi-hop questions.",
        metadata={"topic": "ircot", "source": "internal-kb"},
    ),
    Document(
        page_content="The ReAct pattern (Reason + Act) interleaves LLM reasoning steps with "
                     "tool actions. The model thinks about what to do, acts, observes the result, "
                     "and repeats until it has enough information to answer.",
        metadata={"topic": "react", "source": "internal-kb"},
    ),
]

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
_store = Chroma.from_documents(KB_DOCS, embeddings_model)
_retriever = _store.as_retriever(search_kwargs={"k": 2})

print(f"Knowledge base: {_store._collection.count()} documents indexed")
print("\nTopics available:")
for doc in KB_DOCS:
    print(f"  - {doc.metadata['topic']}: {doc.page_content[:60]}...")

In [ ]:
# ─── 2-B: Verify retrieval before wiring into the agent ───────────────────────
# Always test retrieval in isolation — if it fails here, the agent will also fail.

test_queries = [
    "What is Self-RAG?",
    "How does human-in-the-loop work?",
    "Tell me about adaptive routing",
]

for q in test_queries:
    docs = _retriever.invoke(q)
    print(f"Query: '{q}'")
    for i, doc in enumerate(docs, 1):
        print(f"  [{i}] ({doc.metadata['topic']}) {doc.page_content[:80]}...")
    print()

---

## Part 3 — Defining Tools with `@tool`

---

The `@tool` decorator converts a Python function into a LangChain tool the agent can call.
Two things matter most:

1. **The docstring** — this is the tool description the agent reads to decide *when* to use it.
   Write it for the LLM, not a human developer. Be specific about what the tool returns.

2. **The input schema** — function parameters become the tool's input schema.
   The agent fills these based on the conversation context.

```python
@tool
def my_tool(query: str) -> str:
    """This docstring is the tool description the LLM reads.
    Be specific: what does this tool search? What does it return?"""
    return do_something(query)
```

### Tool Docstring Quality — Examples

| Quality | Docstring example |
|---------|------------------|
| **Poor** | `"Search for information."` — too vague, agent can't distinguish tools |
| **Okay** | `"Search the knowledge base."` — better, but what knowledge base? |
| **Good** | `"Search the internal knowledge base about LangGraph and RAG patterns."` — specific |
| **Best** | `"Search the internal KB for LangGraph and RAG topics. Returns the 2 most relevant chunks. Use for questions about LangGraph nodes, edges, RAG variants, or HITL patterns."` — explicit about when to use it |

In [ ]:
# ─── 3-A: Define the three tools ──────────────────────────────────────────────
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool

_duck = DuckDuckGoSearchResults(max_results=3)


@tool
def vectorstore_search(query: str) -> str:
    """Search the internal knowledge base about LangGraph and RAG patterns.
    Use this for questions about LangGraph, RAG variants (Self-RAG, Adaptive RAG,
    Corrective RAG, IRCoT), human-in-the-loop, and the ReAct pattern."""
    docs = _retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs) or "No results found."


@tool
def web_search(query: str) -> str:
    """Search the web using DuckDuckGo for current, real-time information.
    Use this for recent events, news, current prices, or anything that may
    have changed after the model's knowledge cutoff."""
    return _duck.run(query)


@tool
def calculator(expression: str) -> str:
    """Evaluate a Python math expression and return the result as a string.
    Examples: '2 ** 10', '(5 + 3) * 7', '144 / 12'. Only basic math — no imports."""
    try:
        # __builtins__: {} prevents access to built-in functions like exec, open, import.
        # This is the minimal sandbox for eval() — only arithmetic operators are allowed.
        return str(eval(expression, {"__builtins__": {}}, {}))  # noqa: S307
    except Exception as e:
        return f"Error: {e}"


TOOLS = [vectorstore_search, web_search, calculator]

print("Tools registered:")
for t in TOOLS:
    print(f"  {t.name}: {t.description[:70]}...")

In [ ]:
# ─── 3-B: Test each tool independently before wiring into the agent ────────────
# If a tool fails in isolation, it will fail silently inside the agent loop.
# Always smoke-test tools first.

print("=== vectorstore_search ===")
result_kb = vectorstore_search.invoke({"query": "What is Adaptive RAG?"})
print(result_kb[:200])

print("\n=== calculator ===")
result_calc = calculator.invoke({"expression": "144 / 12 * 7"})
print(result_calc)

print("\n=== calculator (safe — division by zero) ===")
result_err = calculator.invoke({"expression": "1 / 0"})
print(result_err)

print("\n=== calculator (safe — no __import__ allowed) ===")
result_blocked = calculator.invoke({"expression": "__import__('os').system('echo hello')"})
print(result_blocked)

---

## Part 4 — Building the ReAct Agent

---

`create_react_agent` from `langgraph.prebuilt` assembles a full ReAct loop automatically:

```
START
  │
  ▼
[agent node]  ← LLM decides: answer directly, or call a tool?
  │
  ├─── calls tool? ──► [tools node]  ← executes the tool, returns result
  │                          │
  │                          └──► back to [agent node]  (observe + re-think)
  │
  └─── no tool call ──► END
```

You don't write the loop manually — `create_react_agent` builds the graph.
You only provide: the LLM, the tools list, and an optional system prompt.

### System Prompt Design for Tool Selection

The system prompt does two jobs:
1. **Names the tools** and describes when to use each — reinforces the docstrings
2. **Sets the persona** — research assistant, customer support agent, etc.

Without a system prompt, the agent relies only on tool docstrings. That works,
but an explicit system prompt improves routing accuracy (we test this in Part 7).

In [ ]:
# ─── 4-A: Build the agentic RAG agent ─────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = (
    "You are a research assistant with access to three tools:\n"
    "- vectorstore_search: internal knowledge base about LangGraph and RAG patterns\n"
    "- web_search: live web results via DuckDuckGo for current events or recent info\n"
    "- calculator: evaluates Python arithmetic expressions\n\n"
    "Routing rules:\n"
    "  1. For questions about LangGraph, RAG variants, or ReAct → use vectorstore_search first\n"
    "  2. For math and arithmetic → use calculator\n"
    "  3. For current events or information from the last year → use web_search\n"
    "  4. For multi-part questions → call multiple tools in sequence\n"
    "Always ground your final answer in the tool results you received."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(llm, tools=TOOLS, prompt=SYSTEM_PROMPT)

print("Agent ready.")
print(f"Model: gpt-4o-mini (temperature=0 for deterministic tool routing)")
print(f"Tools: {[t.name for t in TOOLS]}")

---

## Part 5 — Running the Agent and Reading the Trace

---

The agent's `invoke()` takes a dict with a `messages` key (list of LangChain messages).
It returns a dict with the same `messages` key extended with all intermediate messages —
the tool calls and tool results are all in the message history.

```python
result = agent.invoke({"messages": [HumanMessage(content="your question")]})
answer = result["messages"][-1].content  # the final AIMessage
```

The full message list is the **trace** — every reasoning step and tool call is recorded.
Reading the trace is the primary debugging technique for agentic systems.

In [ ]:
# ─── 5-A: Run the agent on three different question types ─────────────────────
from langchain_core.messages import HumanMessage

QUESTIONS = [
    ("What is Self-RAG and how does it differ from Corrective RAG?", "→ should use vectorstore"),
    ("What is 144 divided by 12, then multiplied by 7?", "→ should use calculator"),
    ("Who won the most recent FIFA World Cup?", "→ should use web search"),
]

for question, expected_tool in QUESTIONS:
    print(f"\n{'─' * 64}")
    print(f"Q ({expected_tool}):")
    print(f"  {question}")
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    answer = result["messages"][-1].content
    print(f"A: {answer[:300]}")

In [ ]:
# ─── 5-B: Inspect the full ReAct trace — every message in the loop ────────────
# This is the #1 debugging technique for agents.
# The trace shows: HumanMessage → AIMessage (with tool_call) → ToolMessage → AIMessage (final)

trace_question = "What is Adaptive RAG and how does it relate to multi-hop reasoning?"
trace_result = agent.invoke({"messages": [HumanMessage(content=trace_question)]})

print(f"Question: {trace_question}")
print(f"Total messages in trace: {len(trace_result['messages'])}")
print()

for i, msg in enumerate(trace_result["messages"]):
    role = type(msg).__name__
    content_preview = ""

    if hasattr(msg, "content") and msg.content:
        if isinstance(msg.content, str):
            content_preview = msg.content[:150]
        else:
            content_preview = str(msg.content)[:150]

    tool_calls = getattr(msg, "tool_calls", [])

    print(f"[{i}] {role}")
    if tool_calls:
        for tc in tool_calls:
            print(f"     TOOL CALL → {tc['name']}({tc['args']})")
    if content_preview:
        print(f"     {content_preview}")
    print()

---

## Part 6 — Multi-Hop Reasoning

---

**Multi-hop reasoning** occurs when a question requires chaining multiple information sources
or operations to reach an answer. This is where agentic RAG clearly outperforms pipeline RAG.

### Example multi-hop questions

| Question | Hops required |
|----------|---------------|
| "How many characters are in the name of the RAG technique that uses reflection tokens?" | KB lookup → count characters |
| "What year was ReAct published, and how many years ago was that?" | KB lookup → arithmetic |
| "Compare Self-RAG and Corrective RAG, then find the latest paper on either" | KB lookup → KB lookup → web search |

### IRCoT — Interleaving Retrieval with Chain-of-Thought

Trivedi et al. (2022) showed that interleaving retrieval with CoT reasoning dramatically
improves multi-hop performance. The agent essentially implements this automatically:
each tool call is a retrieval step, and the LLM's reasoning between calls is the CoT.

In [ ]:
# ─── 6-A: Multi-hop question — KB lookup + calculator ─────────────────────────
# The agent must: (1) find out what Self-RAG's reflection tokens are,
# then (2) count them — two hops, two tools.

multihop_q = (
    "Self-RAG uses reflection tokens. How many distinct reflection token types does it have, "
    "and what is the square root of that number?"
)

multihop_result = agent.invoke({"messages": [HumanMessage(content=multihop_q)]})

print(f"Q: {multihop_q}")
print()

# Show the reasoning trace in compact form
for msg in multihop_result["messages"]:
    role = type(msg).__name__
    tool_calls = getattr(msg, "tool_calls", [])
    if tool_calls:
        for tc in tool_calls:
            print(f"  ACTION: {tc['name']}('{tc['args'].get('query', tc['args'].get('expression', ''))}')")
    elif role == "ToolMessage":
        content = str(msg.content)[:120]
        print(f"  OBSERVATION: {content}")
    elif role == "AIMessage" and msg.content and not tool_calls:
        print(f"\nFINAL ANSWER: {msg.content[:400]}")

In [ ]:
# ─── 6-B: Multi-hop — compare two KB topics, then check current status online ──
# This tests whether the agent can chain KB + KB + web in sequence.

compare_q = (
    "Briefly compare Self-RAG and Corrective RAG from our knowledge base, "
    "then search the web for the most recent paper on either topic."
)

compare_result = agent.invoke({"messages": [HumanMessage(content=compare_q)]})

print(f"Q: {compare_q}")
print()

# Count tool calls made
tool_call_count = sum(
    len(getattr(m, "tool_calls", []))
    for m in compare_result["messages"]
)
print(f"Tool calls made: {tool_call_count}")
print(f"Total trace messages: {len(compare_result['messages'])}")
print()
print(f"Final answer:\n{compare_result['messages'][-1].content[:500]}")

---

## Part 7 — Debugging and Safety

---

### Common Agentic RAG Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Wrong tool selected** | Calculator called for a KB question | Ambiguous docstrings | Sharpen tool docstrings |
| **Tool called unnecessarily** | Web search fired for a simple math question | Vague system prompt | Add explicit routing rules to system prompt |
| **Infinite loop** | Agent keeps calling tools without concluding | Tool returns "No results" but agent keeps trying | Add max_iterations or return a clear fallback message |
| **Tool ignoring results** | Agent uses its training data instead of tool output | No grounding instruction | Add "Always use tool results" to system prompt |
| **Unsafe eval** | Calculator used to run arbitrary code | No sandbox | Use `{"__builtins__": {}}` — blocks all built-ins |
| **Hallucinated tool arguments** | Agent invents a query that doesn't match the question | Low temperature needed | Set `temperature=0` for tool-routing agents |

### The `eval()` Safety Pattern

```python
# UNSAFE — gives full Python access
eval(expression)  

# SAFE — blocks all built-in functions
eval(expression, {"__builtins__": {}}, {})

# What this blocks:
# __import__('os').system('rm -rf /')  → NameError: __import__ not defined
# open('/etc/passwd')                  → NameError: open not defined
# exec('...')                          → NameError: exec not defined
```

In [ ]:
# ─── 7-A: Ablation — remove system prompt and observe routing quality ──────────
# This tests whether tool docstrings alone are sufficient for correct routing.

agent_no_prompt = create_react_agent(llm, tools=TOOLS)  # no system prompt

test_questions = [
    "What is 7 times 8?",
    "What is Corrective RAG?",
]

print("=== Agent WITH system prompt ===")
for q in test_questions:
    result = agent.invoke({"messages": [HumanMessage(content=q)]})
    tools_used = [
        tc["name"]
        for msg in result["messages"]
        for tc in getattr(msg, "tool_calls", [])
    ]
    print(f"  Q: {q}")
    print(f"  Tools used: {tools_used or ['(no tool — answered directly)']}")
    print()

print("=== Agent WITHOUT system prompt ===")
for q in test_questions:
    result = agent_no_prompt.invoke({"messages": [HumanMessage(content=q)]})
    tools_used = [
        tc["name"]
        for msg in result["messages"]
        for tc in getattr(msg, "tool_calls", [])
    ]
    print(f"  Q: {q}")
    print(f"  Tools used: {tools_used or ['(no tool — answered directly)']}")
    print()

print("Observation: the system prompt reduces misrouting.")
print("Without it, the agent relies entirely on tool docstrings for routing decisions.")

In [ ]:
# ─── 7-B: Verify the calculator sandbox blocks dangerous expressions ───────────

dangerous_expressions = [
    "__import__('os').system('echo PWNED')",
    "open('/etc/passwd').read()",
    "exec('import subprocess; subprocess.run([\"ls\"])')",
    "print('hello')",   # even print() is blocked
]

safe_expressions = [
    "2 ** 32",
    "(100 - 37) * 1.5",
    "144 / 12 * 7",
]

print("=== Dangerous expressions (should all return Error) ===")
for expr in dangerous_expressions:
    result = calculator.invoke({"expression": expr})
    status = "BLOCKED" if result.startswith("Error") else "NOT BLOCKED (security issue!)"
    print(f"  {status}: {expr[:50]}")
    print(f"  Result: {result}")
    print()

print("=== Safe expressions (should all return numbers) ===")
for expr in safe_expressions:
    result = calculator.invoke({"expression": expr})
    print(f"  {expr} = {result}")

---

## Part 8 ★ — Extensions (Bonus)

---

### Adding a Fourth Tool

Any Python function can become a tool. Useful additions:
- `get_current_date()` — returns today's date (so the agent can reason about time)
- `sql_query(sql: str)` — queries a database
- `send_email(to: str, body: str)` — takes an action in the world

### Streaming Agent Responses

Instead of waiting for the full response, stream events as they happen:

```python
for event in agent.stream({"messages": [HumanMessage(content=question)]}):
    if "messages" in event:
        last = event["messages"][-1]
        if hasattr(last, "content") and last.content:
            print(last.content, end="", flush=True)
```

### Adding Memory

By default `create_react_agent` is stateless — each `invoke()` starts fresh.
To add multi-turn memory, use a LangGraph `MemorySaver` checkpointer:

```python
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent = create_react_agent(llm, tools=TOOLS, checkpointer=memory)

config = {"configurable": {"thread_id": "session-123"}}
agent.invoke({"messages": [HumanMessage(content="What is Self-RAG?")]}, config=config)
agent.invoke({"messages": [HumanMessage(content="How does that differ from CRAG?")]}, config=config)
# The second call remembers the first!
```

In [ ]:
# ─── 8-A: Streaming — see events as they happen ───────────────────────────────
print("Streaming agent response for: 'What is ReAct?'\n")
print("─" * 60)

for event in agent.stream({"messages": [HumanMessage(content="What is the ReAct pattern?")]}):
    # Each event is a dict with either 'agent' or 'tools' key
    if "agent" in event:
        msg = event["agent"]["messages"][-1]
        tool_calls = getattr(msg, "tool_calls", [])
        if tool_calls:
            for tc in tool_calls:
                print(f"[TOOL CALL] {tc['name']}({tc['args']})")
        elif msg.content:
            print(f"[FINAL] {msg.content[:300]}")
    elif "tools" in event:
        for tm in event["tools"]["messages"]:
            print(f"[OBSERVATION] {str(tm.content)[:120]}")
    print()

In [ ]:
# ─── 8-B: Adding memory — multi-turn conversation ─────────────────────────────
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent_with_memory = create_react_agent(llm, tools=TOOLS, checkpointer=memory, prompt=SYSTEM_PROMPT)

config = {"configurable": {"thread_id": "workshop-session-1"}}

turn1 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="What is Self-RAG?")]}, config=config
)
print("Turn 1 answer:")
print(turn1["messages"][-1].content[:200])
print()

# Second turn refers back to the first question
turn2 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="How does that differ from Corrective RAG?")]},
    config=config,
)
print("Turn 2 answer (agent remembers the previous exchange):")
print(turn2["messages"][-1].content[:300])

---

## Exercises

---

### Exercise 1 — Add a `get_date` Tool

Create a `@tool` function `get_date()` that takes no arguments and returns today's date as a string
(e.g. `"2025-06-14"`). Add it to the agent and ask `"What day of the week is today?"`
Watch whether the agent uses the tool or guesses from its training data.

**Hint:** Check `datetime.date.today().isoformat()`

---

### Exercise 2 — Sharpen the Tool Docstrings

Rewrite the `vectorstore_search` and `web_search` docstrings to be more specific about
when to use each one. Then run both versions on the same ambiguous question
(e.g. `"Tell me about RAG"`). Does the improved docstring change which tool gets called?

**Hint:** A question like "Tell me about RAG" could go to either vectorstore or web.
Make the docstring say explicitly which one should handle it.

---

### Exercise 3 — Three-Hop Question

Write a question that genuinely requires **three** sequential tool calls to answer.
For example: retrieve a fact, calculate something with it, then search the web for related news.

Run it and count the tool calls in the trace. Does the agent call all three tools in the right order?

---

### Exercise 4 — Add a Fourth Document Source

Add a `document_summary` tool that returns a hardcoded summary of a document
(e.g., a fake company policy). Phrase the docstring so the agent routes policy questions
there instead of to the web. Verify with a policy question.

In [ ]:
# ── Exercise 1 Starter — add get_date tool ────────────────────────────────────
import datetime


@tool
def get_date() -> str:
    """TODO: write a clear docstring explaining when the agent should call this tool."""
    # TODO: return today's date as a string
    pass


# TODO: create a new agent that includes get_date in its tools list
# agent_with_date = create_react_agent(llm, tools=[...], prompt=SYSTEM_PROMPT)

# TODO: ask the agent what day of the week it is
# result = agent_with_date.invoke({"messages": [HumanMessage(content="What day of the week is today?")]})
# print(result["messages"][-1].content)

In [ ]:
# ── Exercise 3 Starter — three-hop question ────────────────────────────────────
# Design a question that requires: vectorstore → calculator → web

# TODO: write your three-hop question
three_hop_q = "TODO: write a question that requires all three tools in sequence"

# three_hop_result = agent.invoke({"messages": [HumanMessage(content=three_hop_q)]})

# TODO: count how many tool calls were made
# tool_calls_made = [
#     tc["name"]
#     for msg in three_hop_result["messages"]
#     for tc in getattr(msg, "tool_calls", [])
# ]
# print(f"Tools called in order: {tool_calls_made}")
# print(f"Answer: {three_hop_result['messages'][-1].content}")

---

## What's Next?

You now have a working agentic RAG system. Here is where to go deeper:

### Improve routing precision
- **Example 25 — Adaptive RAG** ([`../25-adaptive-rag/`](../25-adaptive-rag/)): replace the agent's LLM-driven
  routing with a fast classifier that pre-routes each query to the cheapest correct path.
  Compare latency and cost against the agent-driven approach you just built.

### Add human oversight
- **Example 11 — HITL Approval** ([`../11-hitl-approval/`](../11-hitl-approval/)): interrupt the agent
  before it calls a destructive tool (e.g., send_email, delete_record) and require human approval.
  Useful for production agents that take real-world actions.

### Measure what you built
- **Example 16 — RAG Evaluation** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)):
  score your retrieval quality on Faithfulness, Answer Relevance, and Context Recall using RAGAS.
  The same metrics apply to agentic RAG — just evaluate each tool call independently.

### Go deeper on ReAct and multi-hop
- **Yao et al. (2023) — ReAct**: the paper that defines the Reason+Act loop this notebook implements.
  https://arxiv.org/abs/2210.03629
- **Trivedi et al. (2022) — IRCoT**: Interleaving Retrieval with Chain-of-Thought for multi-hop QA.
  Shows why interleaving (as our agent does) beats retrieve-then-reason.
  https://arxiv.org/abs/2212.10509
- **LangGraph `create_react_agent` source**: see exactly what graph is being built.
  https://github.com/langchain-ai/langgraph/blob/main/libs/langgraph/langgraph/prebuilt/chat_agent_executor.py

### Production concerns
- **Token costs**: every tool call adds latency and cost. Profile your agent with `langsmith` tracing.
- **Max iterations**: set `recursion_limit` on the agent graph to prevent infinite loops.
- **Async**: all LangGraph agents support `ainvoke()` and `astream()` for non-blocking use.

---

*Workshop complete. The natural next step is example 25 (Adaptive RAG) to compare
agent-driven routing against rule-based routing.*

---

## Exercise Answer Key

Attempt the exercises yourself before reading these. The answers below are reference
implementations — not the only correct approach.

### Exercise 1 Answer — `get_date` Tool

**Key insight:** Without the `get_date` tool, the agent either guesses the year from training
data (wrong) or says it doesn't know. With the tool and a clear docstring, it calls `get_date()`
and derives the day of the week from the returned ISO date string.

**What good output looks like:** The trace shows one `get_date` tool call with no arguments,
followed by a final answer that correctly names the day of the week.

In [ ]:
# Answer: Exercise 1 — get_date tool
import datetime


@tool
def get_date() -> str:
    """Returns today's date as an ISO 8601 string (YYYY-MM-DD).
    Use this whenever the user asks about the current date, day of the week,
    or any time-sensitive calculation that requires knowing today's date."""
    return datetime.date.today().isoformat()


SYSTEM_WITH_DATE = SYSTEM_PROMPT + "\n- get_date: returns today's exact date; use for any date/day-of-week questions"

agent_with_date = create_react_agent(
    llm,
    tools=[vectorstore_search, web_search, calculator, get_date],
    prompt=SYSTEM_WITH_DATE,
)

date_result = agent_with_date.invoke({"messages": [HumanMessage(content="What day of the week is today?")]})

# Show tool calls made
tools_called = [
    tc["name"]
    for msg in date_result["messages"]
    for tc in getattr(msg, "tool_calls", [])
]
print(f"Tools called: {tools_called}")
print(f"Answer: {date_result['messages'][-1].content}")

### Exercise 2 Answer — Sharper Docstrings

**Key insight:** For an ambiguous query like "Tell me about RAG", the agent must choose
between `vectorstore_search` (which has internal KB docs) and `web_search`. If both
docstrings mention RAG, the agent may pick either one — or call both.

Sharpening the docstring to say "Use this for questions about *LangGraph-specific* RAG
implementations" vs "Use this for *general* information about RAG from the wider internet"
gives the agent a clear decision boundary.

**What good output looks like:** After the docstring change, the vectorstore handles questions
that match the KB topics, and the web handles general or recent questions — with no overlap.

### Exercise 3 Answer — Three-Hop Question

**Sample three-hop question:**  
"How many RAG variants are described in our knowledge base? Multiply that by 12,
then find recent news about RAG in production."

**Expected trace:**
1. `vectorstore_search("RAG variants")` — retrieves Self-RAG, CRAG, Adaptive RAG, IRCoT
2. `calculator("4 * 12")` — returns 48
3. `web_search("RAG retrieval augmented generation production 2024 2025")` — returns news

**What to check:** The `tools_called` list should have exactly 3 entries in that order.
If the agent calls `vectorstore_search` twice, the question was ambiguous enough
that the agent wanted more context before counting.

In [ ]:
# Answer: Exercise 3 — sample three-hop question
three_hop_q = (
    "How many distinct RAG variants are described in our internal knowledge base? "
    "Then calculate: multiply that count by 12. "
    "Finally, search the web for recent news about RAG systems in production."
)

three_hop_result = agent.invoke({"messages": [HumanMessage(content=three_hop_q)]})

tools_called = [
    tc["name"]
    for msg in three_hop_result["messages"]
    for tc in getattr(msg, "tool_calls", [])
]

print(f"Three-hop question: {three_hop_q[:80]}...")
print(f"Tools called in order: {tools_called}")
print(f"Total tool calls: {len(tools_called)}")
print()
print(f"Final answer:\n{three_hop_result['messages'][-1].content[:400]}")